# Noise-Controlled RD Training

This notebook trains a single Reaction-Diffusion CA model to produce **2 or 3 different textures** based on the noise level:
- **2 textures**: noise=0 → Texture 1, noise=high → Texture 2
- **3 textures**: noise=0 → Texture 1, noise=mid → Texture 2, noise=high → Texture 3

The key mechanism: during training, different pool elements receive different noise levels and are trained against different target textures. The network learns to associate noise level with texture output.

This combines:
- The RD architecture from `rd_training.ipynb` (Laplacian diffusion + learned reaction)
- The Sliced OT loss with rotation invariance
- The multi-texture noise control mechanism from `noise_controlled_nca_training.ipynb`

In [ ]:
# @title Imports and Notebook Utilities
import os
import io
import PIL.Image, PIL.ImageDraw
import base64
import zipfile
import json
import requests
import numpy as np
import matplotlib.pylab as pl
import glob

from IPython.display import Image, HTML, Markdown, clear_output, display
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")

os.environ['FFMPEG_BINARY'] = 'ffmpeg'
import moviepy.editor as mvp
from moviepy.video.io.ffmpeg_writer import FFMPEG_VideoWriter


def imread(url, max_size=None, mode=None):
    if isinstance(url, str) and url.startswith(('http:', 'https:')):
        headers = {
            "User-Agent": "Requests in Colab/0.0 (https://colab.research.google.com/; no-reply@google.com) requests/0.0"
        }
        r = requests.get(url, headers=headers)
        f = io.BytesIO(r.content)
    else:
        f = url
    img = PIL.Image.open(f)
    if max_size is not None:
        img.thumbnail((max_size, max_size), PIL.Image.LANCZOS)
    if mode is not None:
        img = img.convert(mode)
    img = np.float32(img) / 255.0
    return img


def np2pil(a):
    if a.dtype in [np.float32, np.float64]:
        a = np.uint8(np.clip(a, 0, 1) * 255)
    return PIL.Image.fromarray(a)


def imwrite(f, a, fmt=None):
    a = np.asarray(a)
    if isinstance(f, str):
        fmt = f.rsplit('.', 1)[-1].lower()
        if fmt == 'jpg':
            fmt = 'jpeg'
        f = open(f, 'wb')
    np2pil(a).save(f, fmt, quality=95)


def imencode(a, fmt='jpeg'):
    a = np.asarray(a)
    if len(a.shape) == 3 and a.shape[-1] == 4:
        fmt = 'png'
    f = io.BytesIO()
    imwrite(f, a, fmt)
    return f.getvalue()


def im2url(a, fmt='jpeg'):
    encoded = imencode(a, fmt)
    base64_byte_string = base64.b64encode(encoded).decode('ascii')
    return 'data:image/' + fmt.upper() + ';base64,' + base64_byte_string


def imshow(a, fmt='jpeg', id=None):
    return display(Image(data=imencode(a, fmt)), display_id=id)


def grab_plot(close=True):
    """Return the current Matplotlib figure as an image"""
    fig = pl.gcf()
    fig.canvas.draw()
    img = np.array(fig.canvas.renderer._renderer)
    a = np.float32(img[..., 3:] / 255.0)
    img = np.uint8(255 * (1.0 - a) + img[..., :3] * a)  # alpha
    if close:
        pl.close()
    return img


def zoom(img, scale=4):
    img = np.repeat(img, scale, 0)
    img = np.repeat(img, scale, 1)
    return img


class VideoWriter:
    def __init__(self, filename='_autoplay.mp4', fps=30.0, **kw):
        self.writer = None
        self.params = dict(filename=filename, fps=fps, **kw)

    def add(self, img):
        img = np.asarray(img)
        if self.writer is None:
            h, w = img.shape[:2]
            self.writer = FFMPEG_VideoWriter(size=(w, h), **self.params)
        if img.dtype in [np.float32, np.float64]:
            img = np.uint8(img.clip(0, 1) * 255)
        if len(img.shape) == 2:
            img = np.repeat(img[..., None], 3, -1)
        self.writer.write_frame(img)

    def close(self):
        if self.writer:
            self.writer.close()

    def __enter__(self):
        return self

    def __exit__(self, *kw):
        self.close()
        if self.params['filename'] == '_autoplay.mp4':
            self.show()

    def show(self, **kw):
        self.close()
        fn = self.params['filename']
        display(mvp.ipython_display(fn, **kw))

!nvidia-smi -L

In [ ]:
import torch
import torchvision.models as models

torch.set_default_tensor_type('torch.cuda.FloatTensor')

In [ ]:
#@title Loss Function (Sliced OT with Rotation Invariance)
import torch.nn.functional as F
from scipy import ndimage

# Load VGG for feature extraction
vgg = models.vgg16(weights='IMAGENET1K_V1').features

def calc_styles_vgg(imgs, vgg):
    style_layers = [1, 6, 11, 18, 25]
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    x = (imgs - mean) / std
    b, c, h, w = x.shape
    features = [x.reshape(b, c, h * w)]
    for i, layer in enumerate(vgg[:max(style_layers) + 1]):
        x = layer(x)
        if i in style_layers:
            b, c, h, w = x.shape
            features.append(x.reshape(b, c, h * w))
    return features

def project_sort(x, proj):
    return torch.einsum('bcn,cp->bpn', x, proj).sort()[0]

def ot_loss(source, target, proj_n=32):
    ch, n = source.shape[-2:]
    projs = F.normalize(torch.randn(ch, proj_n), dim=0)
    source_proj = project_sort(source, projs)
    target_proj = project_sort(target, projs)
    target_interp = F.interpolate(target_proj, n, mode='nearest')
    return (source_proj - target_interp).square().sum()

def create_rotation_invariant_loss(target_img, num_rotations=64):
    """Create a Sliced OT loss function with rotation invariance.
    
    RD systems are fully isotropic, so we use a variant of texture loss that
    tries to match input image with rotated versions of the target sample.
    """
    # Precompute rotated target features
    target_styles = []
    target_np = target_img[0].permute(1, 2, 0).cpu().numpy()
    
    for r in np.linspace(0.0, 360, num_rotations + 1)[:-1] + 0.12345:
        img_rotated = ndimage.rotate(target_np, r, reshape=False, mode='wrap')
        img_rotated_torch = torch.tensor(img_rotated).permute(2, 0, 1).unsqueeze(0)
        with torch.no_grad():
            style_features = calc_styles_vgg(img_rotated_torch, vgg)
            target_styles.append(style_features)
    
    def loss_f(imgs):
        source_features = calc_styles_vgg(imgs, vgg)
        min_loss = None
        for target_features in target_styles:
            loss = sum(ot_loss(x, y) for x, y in zip(source_features, target_features))
            if min_loss is None:
                min_loss = loss
            else:
                min_loss = torch.minimum(min_loss, loss)
        return min_loss
    
    return loss_f

print("VGG loaded and Sliced OT loss function defined.")

In [ ]:
#@title Load Target Images (2 or 3 textures)
num_textures = 2  #@param [2, 3] {type: "raw"}

from google.colab import files

target_imgs = []
texture_paths = []

for i in range(num_textures):
    if i == 0:
        label = "LOW noise (0.0)"
    elif i == 1 and num_textures == 2:
        label = "HIGH noise"
    elif i == 1:
        label = "MEDIUM noise"
    else:
        label = "HIGH noise"

    print(f"\nUpload TEXTURE {i+1} (appears at {label}):")
    uploaded = files.upload()
    path = list(uploaded.keys())[0]
    texture_paths.append(path)

    img = imread(io.BytesIO(uploaded[path]), max_size=128, mode='RGB')
    target_imgs.append(img)
    print(f"Texture {i+1}:")
    imshow(img)

# Create loss functions for each target (with rotation invariance for RD)
print("\nComputing rotation-invariant loss targets (64 angles per texture)...")
loss_fns = []
for i, img in enumerate(target_imgs):
    tensor = torch.tensor(img).permute(2, 0, 1).unsqueeze(0)
    loss_fn = create_rotation_invariant_loss(tensor)
    loss_fns.append(loss_fn)
    print(f"Created rotation-invariant loss function for texture {i+1}")

print(f"\n✓ Created {num_textures} loss functions with rotation invariance")
print(f"  Texture indices: {list(range(num_textures))}")

In [ ]:
#@title ReactionDiffusionCA Architecture
from scipy.ndimage import gaussian_filter

def laplacian(x):
    """Apply Laplacian filter to all channels independently."""
    b, ch, h, w = x.shape
    # normalized Laplacian (not applying normalization necessitates choosing smaller diffusion constants)
    lap = torch.tensor([[1.0, 2.0, 1.0],
                        [2.0, -12.0, 2.0],
                        [1.0, 2.0, 1.0]])/16. 

    # Depthwise convolution with circular padding
    y = x.reshape(b * ch, 1, h, w)
    y = torch.nn.functional.pad(y, [1, 1, 1, 1], "circular")
    y = torch.nn.functional.conv2d(y, lap[None, None])
    return y.reshape(b, ch, h, w)


class ReactionDiffusionCA(torch.nn.Module):
    def __init__(self, chn=12, hidden_n=128, noise_level=0.1):
        super().__init__()
        self.chn = chn
        self.register_buffer("noise_level", torch.tensor([noise_level]))

        # Reaction network (operates on state directly, not perception)
        # For RD: input is state (chn channels), output is update (chn channels)
        self.w1 = torch.nn.Conv2d(chn, hidden_n, 1, bias=True)
        self.w2 = torch.nn.Conv2d(hidden_n, chn, 1, bias=False)

        # Initialize weights 
        torch.nn.init.xavier_normal_(self.w1.weight, gain=0.1) 
        torch.nn.init.zeros_(self.w1.bias)
        torch.nn.init.zeros_(self.w2.weight)

        # Per-channel diffusion coefficients (multi-scale)
        n_groups = chn // 4
        diff_coef = torch.tensor([0.125, 0.25, 0.5, 1.]).repeat(n_groups)
        # Handle remainder channels
        if chn % 4 != 0:
            diff_coef = torch.cat([diff_coef, torch.ones(chn % 4) * 0.5])
        self.register_buffer("diff_coef", diff_coef)

    def forward(self, x, r=1.0, d=1.0, noise=None, dt=1.0):
        """
        Args:
            x: State tensor [b, chn, h, w]
            r: Reaction rate scaling (default 1.0)
            d: Diffusion rate scaling (default 1.0)
            noise: Noise level to add - can be scalar or per-batch tensor (default None)
            dt: Time step for update (default 1.0)
        """
        # Add noise if specified (supports per-batch noise for noise-controlled training)
        if noise is not None:
            if isinstance(noise, torch.Tensor) and noise.ndim >= 1:
                # Per-batch noise: reshape to [b, 1, 1, 1] for broadcasting
                if noise.ndim == 1:
                    noise = noise.reshape(-1, 1, 1, 1)
                x = x + torch.randn_like(x) * noise
            elif noise > 0:
                x = x + torch.randn_like(x) * noise

        # DIFFUSION TERM: Laplacian with per-channel coefficients
        diff = laplacian(x) * self.diff_coef[None, :, None, None]

        # REACTION TERM: Learned nonlinear dynamics with ReLU activation
        y = self.w1(x)
        y = torch.relu(y)
        react = self.w2(y)

        # Explicit reaction-diffusion update with time step
        x = x + dt * (diff * d + react * r)

        return x

    def seed(self, n, h=128, w=128, seed_type='uniform'):
        """Generate initial state.

        Args:
            n: Batch size
            h, w: Image dimensions
            seed_type: 'uniform' (default NCA-style) or 'gaussian_blobs' (RD-style)
        """
        if seed_type == 'gaussian_blobs':
            return self.seed_gaussian_blobs(n, h, w)
        else:
            # Default: uniform random noise
            return (torch.rand(n, self.chn, h, w) - 0.5) * self.noise_level

    def seed_gaussian_blobs(self, n, h=128, w=128, spot_prob=0.01, spread=3.0):
        """Create seed states with scattered gaussian blobs (from RD paper).

        This creates sparse random spots and blurs them with a Gaussian filter,
        creating smooth blob-like initial conditions. Only RGB channels are
        initialized; hidden channels start at 0.
        """
        # Create sparse random spots (only 0.5% of pixels are 1.0)
        x = np.floor(np.random.uniform(0, 1, (n, h, w, 1)) + spot_prob)

        # Blur with Gaussian filter (mode='wrap' for toroidal boundary)
        x = gaussian_filter(x, sigma=[0.0, spread, spread, 0.0], mode='wrap')

        # Scale by spread^2 to compensate for blur normalization
        x = x * spread ** 2

        # Replicate to RGB channels (3 channels)
        x = np.repeat(x, 3, axis=-1)

        # Pad with zeros for remaining hidden channels
        x = np.pad(x, [(0, 0), (0, 0), (0, 0), (0, self.chn - 3)])

        # Convert to PyTorch tensor and permute to [n, chn, h, w]
        return torch.tensor(x, dtype=torch.float32).permute(0, 3, 1, 2)


def to_rgb(s):
    """Extract RGB channels from state."""
    return s[..., :3, :, :] + 0.5


param_n = sum(p.numel() for p in ReactionDiffusionCA().parameters())
print('ReactionDiffusionCA param count:', param_n)

# Visualize both seed types
print('\nUniform noise seeds (NCA-style):')
img = to_rgb(ReactionDiffusionCA().seed(4, 128, seed_type='uniform'))
imshow(np.hstack(img.permute(0, 2, 3, 1).cpu().numpy()))

print('\nGaussian blob seeds (RD-style):')
img = to_rgb(ReactionDiffusionCA().seed(4, 128, seed_type='gaussian_blobs'))
imshow(np.hstack(img.permute(0, 2, 3, 1).cpu().numpy()))

In [ ]:
#@title Setup Training
import os
import glob
from google.colab import files

# Training hyperparameters - noise levels for each texture
# Automatically space noise levels from 0 to max_noise
max_noise = 0.1  #@param {type: "number"}
noise_levels = [i * max_noise / (num_textures - 1) for i in range(num_textures)]
print(f"Noise levels for {num_textures} textures: {noise_levels}")

# RD-specific hyperparameters
reaction_rate = 1.0    #@param {type: "number"}
diffusion_rate = 1.0   #@param {type: "number"}
dt = 0.5              #@param {type: "number"}
seed_type = "gaussian_blobs"  #@param ["uniform", "gaussian_blobs"]

# Check for existing checkpoints
checkpoint_files = glob.glob('noise_controlled_rd_*.pt')

if checkpoint_files:
    print(f"\nFound {len(checkpoint_files)} checkpoint file(s):")
    for i, f in enumerate(checkpoint_files):
        file_size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  [{i}] {f} ({file_size_mb:.2f} MB)")
    choice = input("\nEnter number to load, 'u' to upload, or Enter for fresh start: ").strip()

    if choice == 'u':
        print("Please upload your checkpoint file:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        checkpoint = torch.load(filename)
    elif choice.isdigit() and 0 <= int(choice) < len(checkpoint_files):
        filename = checkpoint_files[int(choice)]
        print(f'Loading checkpoint from: "{filename}"')
        checkpoint = torch.load(filename)
    else:
        checkpoint = None
else:
    print("No checkpoint files found in Colab storage.")
    upload_choice = input("Upload a checkpoint file? (y/n, default=n): ").strip().lower()
    if upload_choice == 'y':
        print("Please upload your checkpoint file:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        checkpoint = torch.load(filename)
    else:
        checkpoint = None

# Initialize model
model = ReactionDiffusionCA()

if checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    start_iter = checkpoint.get('iteration', 0) + 1
    loss_log = checkpoint.get('loss_log', [])
    pool = checkpoint.get('pool', model.seed(256, seed_type=seed_type))
    print(f"Resumed from iteration {start_iter - 1}")
else:
    start_iter = 0
    loss_log = []
    with torch.no_grad():
        pool = model.seed(256, seed_type=seed_type)
    print(f"Starting fresh training with seed_type='{seed_type}'")

# Optimizer with adaptive LR
opt = torch.optim.Adam(model.parameters(), 1e-3, capturable=True)
lr_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt, mode='min', factor=0.2, patience=1000,
    threshold=0.01, threshold_mode='rel', min_lr=1e-6
)

if checkpoint and 'optimizer_state_dict' in checkpoint:
    opt.load_state_dict(checkpoint['optimizer_state_dict'])
if checkpoint and 'scheduler_state_dict' in checkpoint:
    lr_sched.load_state_dict(checkpoint['scheduler_state_dict'])

print(f"\nPool shape: {pool.shape}")
print(f"Training {num_textures} textures with noise levels: {noise_levels}")
print(f"RD parameters: dt={dt}, reaction_rate={reaction_rate}, diffusion_rate={diffusion_rate}")

In [ ]:
#@title Training Loop {vertical-output: true}

num_iterations = 15000  #@param {type: "integer"}
batch_size = 4  #@param {type: "integer"}

try:
    for i in range(start_iter, start_iter + num_iterations):
        with torch.no_grad():
            # Sample batch indices from pool
            batch_idx = np.random.choice(len(pool), batch_size, replace=False)
            s = pool[batch_idx]

            # Inject fresh seed periodically (less often for RD)
            if i % 32 == 0:
                s[:1] = model.seed(1, seed_type=seed_type)

            # KEY: Map batch indices to texture indices (0, 1, or 2)
            texture_indices = batch_idx % num_textures

            # Get noise level for each batch element based on its target texture
            batch_noise = torch.tensor(
                [noise_levels[t] for t in texture_indices],
                device='cuda', dtype=torch.float32
            )

        # Run forward steps with noise-controlled texture selection
        step_n = np.random.randint(32, 96)
        for k in range(step_n):
            s = model(s, r=reaction_rate, d=diffusion_rate, noise=batch_noise, dt=dt)

        # Compute loss - each batch element uses its corresponding target
        overflow_loss = (s - s.clamp(-1.0, 1.0)).abs().sum()

        # Sum losses from appropriate loss functions
        loss = overflow_loss
        rgb = to_rgb(s)
        for b in range(batch_size):
            t_idx = texture_indices[b]
            loss = loss + loss_fns[t_idx](rgb[b:b+1])

        # Backward and optimize
        loss.backward()
        for p in model.parameters():
            p.grad /= (p.grad.norm() + 1e-8)  # Normalize gradients
        opt.step()
        opt.zero_grad()

        with torch.no_grad():
            lr_sched.step(loss)
            pool[batch_idx] = s.detach()
            loss_log.append(loss.item())

            # Display progress
            if i % 10 == 0:
                lr = opt.param_groups[0]['lr']
                display(Markdown(f"iter: {i}, loss: {loss.item():.2e}, lr: {lr:.2e}"), display_id='stats')

            # Visualize
            if i % 20 == 0:
                # Show loss plot and current batch
                pl.figure(figsize=(12, 3))
                
                pl.subplot(1, 2, 1)
                pl.plot(loss_log, '.', alpha=0.1)
                pl.yscale('log')
                pl.title('Loss')
                pl.xlabel('Iteration')

                # Show current batch with texture labels
                pl.subplot(1, 2, 2)
                imgs = rgb.permute(0, 2, 3, 1).cpu().numpy()
                labels = [f'T{t+1}' for t in texture_indices]
                pl.imshow(np.hstack(imgs))
                pl.title(' | '.join(labels))
                pl.axis('off')

                pl.tight_layout()
                imshow(grab_plot(), id='progress')

            # Save checkpoint
            if i % 2000 == 0 and i > start_iter:
                checkpoint = {
                    'iteration': i,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': opt.state_dict(),
                    'scheduler_state_dict': lr_sched.state_dict(),
                    'loss_log': loss_log,
                    'pool': pool,
                    'noise_levels': noise_levels,
                    'num_textures': num_textures,
                    'dt': dt,
                    'reaction_rate': reaction_rate,
                    'diffusion_rate': diffusion_rate,
                    'seed_type': seed_type,
                }
                torch.save(checkpoint, f'noise_controlled_rd_iter_{i}.pt')
                print(f"\nCheckpoint saved at iteration {i}")

except KeyboardInterrupt:
    print('\n\nTraining interrupted by user!')
    print(f'Saving checkpoint at iteration {i}...')
    checkpoint = {
        'iteration': i,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': opt.state_dict(),
        'scheduler_state_dict': lr_sched.state_dict(),
        'loss_log': loss_log,
        'pool': pool,
        'noise_levels': noise_levels,
        'num_textures': num_textures,
        'dt': dt,
        'reaction_rate': reaction_rate,
        'diffusion_rate': diffusion_rate,
        'seed_type': seed_type,
    }
    torch.save(checkpoint, f'noise_controlled_rd_interrupted_iter_{i}.pt')
    print(f'Checkpoint saved!')

print(f'\nTraining completed at iteration {i}')

In [ ]:
#@title Test Noise Control

# Generate samples across the noise range
max_test_noise = max(noise_levels) * 1.5  # Go beyond training range
test_noise_levels = np.linspace(0, max_test_noise, 8)

with torch.no_grad():
    results = []
    for nl in test_noise_levels:
        s = model.seed(1, seed_type=seed_type)
        for _ in range(64):
            s = model(s, r=reaction_rate, d=diffusion_rate, noise=nl, dt=dt)
        results.append(to_rgb(s)[0].permute(1, 2, 0).cpu().numpy())

    # Display
    fig, axes = pl.subplots(1, len(test_noise_levels), figsize=(16, 2))
    for ax, img, nl in zip(axes, results, test_noise_levels):
        ax.imshow(img)
        ax.set_title(f'{nl:.3f}')
        ax.axis('off')

    # Add markers for training noise levels
    noise_str = ', '.join([f'{n:.3f}' for n in noise_levels])
    pl.suptitle(f'Noise Level → Texture Transition\nTraining levels: [{noise_str}]')
    pl.tight_layout()
    imshow(grab_plot())

# Show target textures for comparison
print("\nTarget textures for reference:")
fig, axes = pl.subplots(1, num_textures, figsize=(4 * num_textures, 4))
if num_textures == 1:
    axes = [axes]
for i, (ax, img) in enumerate(zip(axes, target_imgs)):
    ax.imshow(img)
    ax.set_title(f'Texture {i+1} (noise={noise_levels[i]:.3f})')
    ax.axis('off')
pl.tight_layout()
imshow(grab_plot())

In [ ]:
#@title Save Model Weights

# Save for demo
weights_file = 'noise_controlled_rd_weights.pt'

# Include training params for conversion script
state_dict = model.state_dict()
state_dict['_training_params'] = {
    'noise_levels': noise_levels,
    'num_textures': num_textures,
    'model_type': 'rd',
    'dt': dt,
    'noise_level': max(noise_levels),  # For demo compatibility
    'diff_coef': model.diff_coef.cpu().tolist(),
    'reaction_rate': reaction_rate,
    'diffusion_rate': diffusion_rate,
    'seed_type': seed_type,
    'texture_paths': texture_paths,
}

torch.save(state_dict, weights_file)
print(f"Saved: {weights_file}")
print(f"  Model type: rd")
print(f"  Textures: {num_textures}")
print(f"  Noise levels: {noise_levels}")
print(f"  dt: {dt}")
print(f"  diff_coef: {model.diff_coef.cpu().tolist()}")
print(f"  reaction_rate: {reaction_rate}")
print(f"  diffusion_rate: {diffusion_rate}")
print(f"  seed_type: {seed_type}")

# Also save full checkpoint
checkpoint_file = 'noise_controlled_rd_final.pt'
checkpoint = {
    'iteration': i,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': opt.state_dict(),
    'scheduler_state_dict': lr_sched.state_dict(),
    'loss_log': loss_log,
    'pool': pool,
    'noise_levels': noise_levels,
    'num_textures': num_textures,
    'dt': dt,
    'reaction_rate': reaction_rate,
    'diffusion_rate': diffusion_rate,
    'seed_type': seed_type,
}
torch.save(checkpoint, checkpoint_file)
print(f"\nFull checkpoint saved: {checkpoint_file}")

# Download
from google.colab import files
files.download(weights_file)